In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/libri",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import sclite_trn, parse_sclite_summary
from sj_utils.evaluator import TimeChecker

In [ ]:
from util import get_faster_whisper_transcriber, normalize_text

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
_transcriber = get_faster_whisper_transcriber(model, SAMPLE_RATE)
transcriber = lambda audio: _transcriber(audio, transcribe_time)

In [ ]:
processed_time.start()
data = search_all_ref_and_hyp(src, transcriber, normalize_text, 2)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}